# Week 5 — Member 8: First Model Call, Timing and JSONL Logging

**Learning roadmap topic:** LLM APIs and prompt design  
**Assigned chunk:** A first model call with a provider SDK: record input, output and latency  
**Friday task:** Wrap the model call with timing and a JSON log record.

This notebook reconstructs the lessons taught from **Review Learning Roadmap (11)** through the current conversation. It follows the same beginner-friendly sequence: understand the idea, inspect an example, run code, observe the result, and check understanding.

## What you should be able to do

By the end of the material, you should be able to:

1. Explain what an AI model, Ollama, an API, HTTP, JSON and an endpoint are.
2. Describe how a model request is sent and how its response is received.
3. Make a model call from Python.
4. extract the model's answer and token usage.
5. Measure the time taken by a request.
6. save each call as one JSON record in a `.jsonl` file.
7. Handle connection, timeout and HTTP status errors.
8. Explain the difference between calling an API directly with `httpx` and using a provider's SDK.

> **Important distinction:** The working implementation in this lesson used Groq through the `httpx` HTTP client. That successfully taught model calling, timing, token extraction and logging, but it was **not yet a provider-SDK implementation**. The provider SDK is explained near the end, and using it is the next learning step.

## Lesson map and progress

| Section | Topic | Status from the lessons |
|---|---|---|
| 1 | Assignment, team decision and provider setup | Covered |
| 2 | AI models, Ollama and local vs hosted models | Covered |
| 3 | APIs, requests, responses and endpoints | Covered |
| 4 | JSON and Python dictionaries | Covered |
| 5 | HTTP, POST and `httpx` | Covered |
| 6 | Ollama chat request and response structure | Covered conceptually |
| 7 | Tokens and token usage | Covered |
| 8 | Latency and timing | Covered |
| 9 | Reading API responses in Python | Covered |
| 10 | Reusable Python functions | Covered |
| 11 | JSONL logging | Covered |
| 12 | Error handling | Covered and tested |
| 13 | Multiple calls and log verification | Covered and tested |
| 14 | Provider SDK meaning and comparison | Explained; SDK implementation remains |

The progress labels distinguish between what we actually practised and what we only discussed. Do not describe the Groq `httpx` version as an SDK implementation.

# Section 1 — Understand the assignment and our provider decision

## 1.1 What the group assignment says

Your assigned chunk is:

**A first model call with a provider SDK: record input, output and latency.**

The learning outcome is to make a model call and log its input, output, latency and token use. The Friday task is to wrap the model call with timing and a JSON log record.

Your part is focused on making one provider call and recording what happened. It is not the same as Member 1's broader provider abstraction task, which involves building a common interface with multiple provider adapters.

## 1.2 The team's provider decision

The team agreed to use **Ollama** for the project. The team's guide selected:

- `qwen2.5:7b` as the default model.
- `llama3.1:8b` as a comparison model.

You were working in Windows PowerShell and were not ready to install Ollama locally on that PC. The guide estimated roughly 10 GB for the two model downloads and recommended 16 GB RAM.

The team guide:
[Harbour Ollama guide](https://github.com/Snrboi/harbour/blob/learn/ollama_guide.md)

## 1.3 Why we practised with Groq temporarily

You created an Ollama Cloud API key, but the attempted cloud request using `gemma4:31b` returned **402 Payment Required**. Seeing a model in a model list does not necessarily mean the account can use it without charge.

Rather than block the learning exercise, you signed up for Groq and used it temporarily to practise making a hosted model request.

The working Groq model used in the lesson was:

`openai/gpt-oss-20b`

**Keep the distinction clear:**
- Ollama is the team's intended provider for the project.
- Groq was the temporary learning provider used for successful tests.
- Practising with Groq does not mean the team changed its provider decision.

# Section 2 — AI models, Ollama and hosted models

## 2.1 What is an AI model?

An AI model is software trained to recognise patterns and generate outputs from inputs. A chat model can receive a message and generate a response.

For our project:
- **Input:** a customer email or support message.
- **Model processing:** the model interprets the message and generates text.
- **Output:** the model's response.

The model does not automatically know the real state of a customer's account, billing system or support platform. It only knows the information supplied to it and what it learned during training. An application must perform real account actions itself.

## 2.2 What is Ollama?

Ollama is software for running and managing AI models. It can run models locally on a computer and also provides ways to access hosted models.

Examples of model names discussed:
- `qwen2.5:7b`
- `llama3.1:8b`
- `gemma4:31b` (the attempted Ollama Cloud model in the lesson)

## 2.3 Local vs hosted models

| Local model | Hosted model |
|---|---|
| Runs on your own computer | Runs on a provider's infrastructure |
| Requires local hardware resources and model downloads | Requires internet access and provider access |
| You communicate with a local service | You communicate with a remote service |
| Example local Ollama address: `http://localhost:11434` | Example Groq API address: `https://api.groq.com/openai/v1/chat/completions` |

**Check your understanding:** If a model runs on a provider's server and your Python program sends it a request over the internet, is that a local or hosted model?

**Answer:** Hosted.

# Section 3 — APIs, requests, responses and endpoints

## 3.1 What is an API?

API stands for **Application Programming Interface**. It is a defined way for one piece of software to communicate with another.

Imagine a restaurant:
- You (the client application) make an order.
- The waiter carries the order to the kitchen.
- The kitchen prepares it.
- The waiter brings back the result.

The API is like the agreed communication process between the client and the service.

## 3.2 Request and response

An API interaction usually has two important parts:

1. **Request:** the information your application sends.
2. **Response:** the information the service sends back.

In our model project:
- Python sends the model name and messages.
- The model provider processes the request.
- The provider returns the generated answer and usage information.

## 3.3 What is an endpoint?

An endpoint is a particular address where an API accepts a particular kind of request.

Examples from the lesson:

| Provider/API | Endpoint |
|---|---|
| Ollama local chat API | `http://localhost:11434/api/chat` |
| Groq chat completions API | `https://api.groq.com/openai/v1/chat/completions` |

An endpoint is not the same thing as the entire API. It is a specific route within the API.

## 3.4 What is HTTP?

HTTP stands for **Hypertext Transfer Protocol**. It is a common protocol used by clients and servers to communicate.

Useful terms:
- **Client:** the program making the request; here, our Python script.
- **Server:** the computer/service receiving and processing the request.
- **Host:** the server name or address.
- **Port:** a numbered communication endpoint on a computer.
- **Path:** the route after the host and port, such as `/api/chat`.

`localhost` means the computer running the program. `127.0.0.1` is the loopback IP address for that same computer.

## 3.5 HTTP methods

An HTTP method describes the kind of operation being requested.

- `GET` commonly retrieves information.
- `POST` commonly sends information for processing or creation.
- `PUT` commonly replaces a resource.
- `PATCH` commonly updates part of a resource.
- `DELETE` commonly removes a resource.

We used **POST** because we were sending a message and request data to a model endpoint.

# Section 4 — JSON and Python dictionaries

## 4.1 What is JSON?

JSON stands for **JavaScript Object Notation**. It is a text format used to represent structured data and exchange it between applications.

A simple JSON object:

```json
{
  "name": "Ada",
  "age": 25,
  "active": true
}
```

JSON uses key-value pairs. A key is the label; the value is the information stored under that label.

Common JSON value types:
- String: `"hello"`
- Number: `42`
- Boolean: `true` or `false`
- Null: `null`
- Array: `["red", "blue"]`
- Object: `{"city": "Lagos"}`

## 4.2 Python dictionaries

A Python dictionary is also a collection of key-value pairs:

```python
person = {
    "name": "Ada",
    "age": 25,
    "active": True
}

print(person["name"])
```

Output:

```text
Ada
```

A Python dictionary is not itself JSON text. It is a Python data structure. Python libraries can convert between dictionaries and JSON.

## 4.3 Convert between Python and JSON

```python
import json

person = {
    "name": "Ada",
    "age": 25
}

# Convert a Python object into JSON text
json_text = json.dumps(person)
print(json_text)

# Convert JSON text into a Python object
decoded_person = json.loads(json_text)
print(decoded_person["name"])
```

- `json.dumps()` converts a Python object into a JSON-formatted string.
- `json.loads()` parses JSON text into a Python object.

This distinction becomes important when sending a request and processing a response.

# Section 5 — Sending HTTP requests with `httpx`

## 5.1 What is `httpx`?

`httpx` is a Python library that lets your program send HTTP requests and receive HTTP responses.

We used it to communicate with the Groq API directly, without using Groq's provider SDK.

The general pattern is:

```python
response = httpx.post(
    "https://example.com/api/endpoint",
    headers=headers,
    json=request_data,
    timeout=60.0
)
```

What the arguments mean:
- The first argument is the endpoint URL.
- `headers` carries request metadata, including authentication.
- `json` supplies the request body as Python data; `httpx` serialises it as JSON.
- `timeout` sets how long the client waits for the request.

## 5.2 HTTP response object vs decoded JSON

The result of `httpx.post()` is an HTTP response object. It is not yet the Python dictionary containing the API's JSON data.

Use:

```python
data = response.json()
```

to parse the response body as JSON.

We also used:

```python
response.raise_for_status()
```

This checks the HTTP status. If the server returns an unsuccessful status, it raises an `httpx.HTTPStatusError`.

# Section 6 — Understanding the Ollama chat request and response

We first learned the structure of an Ollama chat request conceptually. The local Ollama endpoint discussed was:

`http://localhost:11434/api/chat`

## 6.1 Example Ollama request

```json
{
  "model": "qwen2.5:7b",
  "messages": [
    {
      "role": "user",
      "content": "Classify this email: I cannot access my account."
    }
  ],
  "stream": false
}
```

What each field means:

- `model`: which model Ollama should use.
- `messages`: the conversation messages sent to the model.
- `role`: identifies who is speaking.
- `content`: the actual text of the message.
- `stream: false`: asks for the complete response in one result rather than a stream of partial chunks.

## 6.2 Message roles

The roles discussed were:

- `system`: instructions that guide the model's behaviour.
- `user`: the message or question supplied by the user.
- `assistant`: a message generated by the model.

## 6.3 Example Ollama response fields

The Ollama response structure differs from Groq's response structure. In the Ollama format discussed:

- `data["message"]["content"]` contains the generated text.
- `data["prompt_eval_count"]` reports input tokens.
- `data["eval_count"]` reports output tokens.
- `data["total_duration"]` reports total provider-side duration in nanoseconds.

**Important:** Do not use Ollama response field names when processing a Groq response. Each provider/API can have its own response schema.

# Section 7 — Tokens and token usage

## 7.1 What is a token?

A token is a unit of text processed by a language model. A token is not necessarily one whole word. Depending on the tokenizer, a word may be one token or split into multiple tokens. Punctuation and parts of words can also be tokens.

## 7.2 Input and output tokens

- **Input tokens:** the text sent to the model, including relevant instructions and messages.
- **Output tokens:** the text generated by the model.
- **Total tokens:** commonly the sum of input and output tokens for a call.

Token usage matters because:
- Hosted model providers may charge based on input and output token usage.
- Models have context limits.
- Tracking tokens helps estimate usage and compare calls.

For a local Ollama model, there may be no provider per-token API charge, but measuring token use is still useful.

## 7.3 Provider-specific token fields

Ollama fields discussed:
- `prompt_eval_count`: input tokens.
- `eval_count`: output tokens.

Groq response fields used in our successful code:
- `usage.prompt_tokens`: input tokens.
- `usage.completion_tokens`: output tokens.
- `usage.total_tokens`: total tokens.

The names differ, even though the concepts are similar.

# Section 8 — Latency and timing

## 8.1 What is latency?

Latency is the time taken for an operation to complete. For a model API call, it is useful to measure how long your program waits for the request and response.

A slow response may affect the user experience of an application.

## 8.2 Measure elapsed time in Python

We used `time.perf_counter()`, a high-resolution timer designed for measuring elapsed time.

```python
import time

start_time = time.perf_counter()

# Perform the operation being measured here

end_time = time.perf_counter()
latency_seconds = end_time - start_time

print(latency_seconds)
```

The formula is:

`latency = end time - start time`

## 8.3 Client-side vs provider-side timing

Our Python timer measures the elapsed time around the HTTP request and response processing in our script. Provider-side timing fields, when available, measure time inside the provider's service.

For Ollama, timing fields such as `total_duration`, `load_duration`, `prompt_eval_duration` and `eval_duration` are reported in nanoseconds.

Do not confuse nanoseconds with seconds. One second is one billion nanoseconds.

## 8.4 Example from the successful Groq tests

One successful call reported:
- Input tokens: 84
- Output tokens: 382
- Total tokens: 466
- Client-measured latency: about 1.72 seconds

Another call reported about 1.10 seconds. Repeated calls can have different latency due to network and service conditions.

# Section 9 — Extracting information from a response

## 9.1 Nested dictionaries

An API response often contains dictionaries inside other dictionaries.

For example:

```python
data = {
    "message": {
        "content": "Hello!"
    }
}
```

To access the content:

```python
output = data["message"]["content"]
print(output)
```

Read this from the outside in:
1. Look up the `"message"` key.
2. Inside that dictionary, look up `"content"`.

## 9.2 Groq response structure used in this lesson

The Groq-compatible chat response was read using:

```python
output = data["choices"][0]["message"]["content"]

usage = data["usage"]
input_tokens = usage["prompt_tokens"]
output_tokens = usage["completion_tokens"]
total_tokens = usage["total_tokens"]
```

Explanation:
- `choices` is a list of generated choices.
- `[0]` selects the first choice.
- `message` contains the assistant message.
- `content` contains the generated text.
- `usage` contains token counts.

This is different from the Ollama response path, `data["message"]["content"]`.

# Section 10 — Reusable Python functions

## 10.1 Why use a function?

A function is a named, reusable block of code. Instead of copying the same request logic every time, we define it once and call it with different inputs.

## 10.2 Function vocabulary

```python
def greet(name):
    return "Hello, " + name

message = greet("Ada")
print(message)
```

- `def` starts a function definition.
- `greet` is the function name.
- `name` is a parameter: a variable that receives an input.
- `"Ada"` is an argument: the actual value passed to the function.
- `return` sends a result back to the code that called the function.
- `print()` displays something in the terminal; it does not replace `return`.

## 10.3 First model-call function concept

The first simple Ollama-style function was:

```python
import httpx

def call_model(user_input):
    request_data = {
        "model": "qwen2.5:7b",
        "messages": [
            {
                "role": "user",
                "content": user_input
            }
        ],
        "stream": False
    }

    response = httpx.post(
        "http://localhost:11434/api/chat",
        json=request_data
    )

    data = response.json()
    return data
```

This function accepts an input message, prepares the request, sends it, parses the JSON response and returns the decoded data.

Later, the working implementation was adapted to Groq's endpoint and response schema.

# Section 11 — JSONL logging

## 11.1 What is logging?

Logging means keeping a record of what a program did. In our project, we want to retain details about each model call so that we can inspect or analyse calls later.

The log record should include:
- model name
- input
- output
- input token count
- output token count
- total token count
- latency in seconds

## 11.2 A Python dictionary as a log record

```python
log_record = {
    "model": "openai/gpt-oss-20b",
    "input": "I cannot access my account.",
    "output": "Please tell me what error you see.",
    "input_tokens": 12,
    "output_tokens": 10,
    "total_tokens": 22,
    "latency_seconds": 1.25
}
```

This is a Python dictionary.

## 11.3 What is JSONL?

JSONL means **JSON Lines**. It is a text-file format where each line contains one complete JSON value, usually one JSON object.

Example file contents:

```jsonl
{"model":"model-a","input":"Hello","output":"Hi","latency_seconds":0.8}
{"model":"model-a","input":"How are you?","output":"I'm well","latency_seconds":1.1}
```

Each line is an independent record. This is useful for appending new model-call records.

## 11.4 Append one record to the file

```python
import json

with open("model_calls.jsonl", "a", encoding="utf-8") as file:
    file.write(json.dumps(log_record) + "\n")
```

Explanation:
- `open()` opens a file.
- `"a"` means append mode: add to the end without deleting existing records.
- `encoding="utf-8"` supports a wide range of characters.
- `with` ensures the file is properly closed after the block finishes.
- `json.dumps(log_record)` converts the Python dictionary to JSON text.
- `+"\n"` adds a newline so the next record starts on a new line.

Do not confuse:
- `"a"` = append.
- `"w"` = write, which normally replaces the existing file contents.

## 11.5 Read the saved records

```python
import json

with open("model_calls.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        record = json.loads(line)
        print(json.dumps(record, indent=4))
```

- `"r"` opens the file in read mode.
- The loop reads one line at a time.
- `json.loads(line)` parses that JSON line into a Python dictionary.
- `indent=4` makes the displayed JSON easier to read.

The user verified that the log file contained records from multiple calls. The escaped characters you may see, such as `\u2019` or `\n`, are normal JSON representations of characters and line breaks.

# Section 12 — Environment variables and protecting API keys

## 12.1 Why API keys matter

An API key is a secret credential used to identify or authorise an application. Anyone who obtains an active key may be able to use the associated service.

Never:
- paste an API key into a public notebook or GitHub repository;
- share your `.env` file;
- hard-code a real key into source code;
- include a real key in screenshots or teaching materials.

## 12.2 The setup used in the lesson

The project directory was:

`C:\Users\BESTMAN\my-git-repo`

The user was working in Windows PowerShell with a Python virtual environment named `.venv`.

The required packages installed in that environment were:
- `httpx`
- `python-dotenv`

The `.env` file contained the keys as environment variables:

```text
OLLAMA_API_KEY=your_ollama_key_here
GROQ_API_KEY=your_groq_key_here
```

Those are placeholders for teaching only. Do not put real keys in this notebook.

The `.gitignore` file contained:

```text
.env
```

This tells Git to ignore the `.env` file.

## 12.3 Load the key in Python

```python
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("GROQ_API_KEY")

if not API_KEY:
    raise ValueError("Groq API key was not found.")
```

Explanation:
- `os` provides access to operating-system environment variables.
- `load_dotenv()` reads variables from the `.env` file.
- `os.getenv("GROQ_API_KEY")` retrieves the variable.
- The `if not API_KEY` check stops the program with a clear message if the key is missing.

The key itself should never be printed.

# Section 13 — The working Groq implementation using `httpx`

This is the consolidated working implementation from the lesson. It makes a Groq chat completion request, measures elapsed time, extracts the response and token usage, creates a log record, and appends the record to `model_calls.jsonl`.

**Important:** This is a direct HTTP implementation using `httpx`, not the Groq provider SDK.

Before running it:
1. Ensure your virtual environment is active.
2. Ensure `httpx` and `python-dotenv` are installed in that environment.
3. Ensure your `.env` file contains `GROQ_API_KEY`.
4. Never paste the actual key into this notebook.

In [ ]:
import os
import time
import json
import httpx
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

API_KEY = os.getenv("GROQ_API_KEY")

if not API_KEY:
    raise ValueError("Groq API key was not found.")


def call_model(user_input):
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    # Prepare the request body
    request_data = {
        "model": "openai/gpt-oss-20b",
        "messages": [
            {
                "role": "user",
                "content": user_input
            }
        ]
    }

    try:
        # Start measuring elapsed time
        start_time = time.perf_counter()

        # Send the request to Groq
        response = httpx.post(
            "https://api.groq.com/openai/v1/chat/completions",
            headers=headers,
            json=request_data,
            timeout=60.0
        )

        # Raise an exception for unsuccessful HTTP status codes
        response.raise_for_status()

        # Decode the JSON response into a Python dictionary
        data = response.json()

        # Stop measuring time
        end_time = time.perf_counter()
        latency = end_time - start_time

        # Extract the model's generated answer
        output = data["choices"][0]["message"]["content"]

        # Extract token usage
        usage = data["usage"]
        input_tokens = usage["prompt_tokens"]
        output_tokens = usage["completion_tokens"]
        total_tokens = usage["total_tokens"]

        # Build one log record
        log_record = {
            "model": data["model"],
            "input": user_input,
            "output": output,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": total_tokens,
            "latency_seconds": latency
        }

        # Append the record as one JSON line
        with open("model_calls.jsonl", "a", encoding="utf-8") as file:
            file.write(json.dumps(log_record) + "\n")

        # Return the record so the caller can use it
        return log_record

    except httpx.ConnectError:
        print("Could not connect to Groq.")

    except httpx.TimeoutException:
        print("The request took too long.")

    except httpx.HTTPStatusError as error:
        print("Groq returned an HTTP error:")
        print("Status:", error.response.status_code)
        print("Details:", error.response.text)

    except (KeyError, IndexError, TypeError, ValueError) as error:
        print("Could not process the model response:", error)

    return None

## 13.1 Run one successful model call

Run the following cell only when your `.env` file and internet connection are ready. It sends a real request and may use your provider account's quota.

In [ ]:
result = call_model(
    "My account is locked. I cannot log in."
)

if result is not None:
    print("\nMODEL OUTPUT:")
    print(result["output"])

    print("\nTOKEN USAGE:")
    print("Input tokens:", result["input_tokens"])
    print("Output tokens:", result["output_tokens"])
    print("Total tokens:", result["total_tokens"])

    print("\nLATENCY:")
    print(result["latency_seconds"], "seconds")

    print("\nLog successfully saved.")

## 13.2 What the successful run showed

One successful run returned:
- Model: `openai/gpt-oss-20b`
- Input tokens: 84
- Output tokens: 382
- Total tokens: 466
- Latency: approximately 1.72 seconds

A later run returned a different answer and different usage/latency values. This is normal: model output and response time can vary between calls.

The model's generated answer is not proof that an action was performed. During testing, one response claimed it could send a password-reset link and another invented a service name. Our application must not claim to unlock accounts, issue refunds or send emails unless the application actually performs those actions through trusted systems.

# Section 14 — Multiple calls and checking the JSONL file

## 14.1 Why test multiple calls?

A single successful record is not enough to prove that append logging works. We tested several inputs in one run:

1. `My account is locked. I cannot log in.`
2. `I was charged twice for my subscription.`
3. `I want to know how to change my registered email address.`

The same `call_model()` function was reused for all three messages.

In [ ]:
test_messages = [
    "My account is locked. I cannot log in.",
    "I was charged twice for my subscription.",
    "I want to know how to change my registered email address."
]

for message in test_messages:
    result = call_model(message)

    if result is not None:
        print("\nINPUT:", result["input"])
        print("OUTPUT:", result["output"])
        print("LATENCY:", result["latency_seconds"])
        print("-" * 50)

## 14.2 Count the saved records

Each successful model call should append one JSON object on its own line.

In [ ]:
print("TOTAL SAVED RECORDS:")

with open("model_calls.jsonl", "r", encoding="utf-8") as file:
    records = file.readlines()

print(len(records))

## 14.3 Inspect the saved records

In [ ]:
print("SAVED LOG RECORDS:")

with open("model_calls.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        record = json.loads(line)
        print(json.dumps(record, indent=4))

**What was verified:** the user confirmed that all the logs were present in the JSONL file. This showed that successful calls were being appended and could be read back.

The exact record count depends on how many times the script has been run. Do not assume the file contains only the records from the most recent test.

# Section 15 — Error handling

An API request can fail. A robust program should handle expected failures and provide useful information rather than unexpectedly terminating.

The implementation included three HTTP-related error handlers and a response-parsing handler.

## 15.1 What is an exception?

An exception is an error event raised while a program is running. Python can handle it using `try` and `except`.

- `try`: run code that might fail.
- `except`: handle a particular kind of failure.
- `return None`: indicate that the function did not produce a successful result.

## 15.2 Connection error — `httpx.ConnectError`

This means Python could not establish a connection to the server.

We tested this by temporarily changing the endpoint to:

`http://127.0.0.1:1/api/chat`

`127.0.0.1` refers to the local computer. Port `1` was chosen as a test destination where no API server was expected to be listening.

The terminal showed:

```text
Could not connect to Groq.

TEST RESULT: None
```

This confirmed that the connection error was caught and the function returned `None`.

## 15.3 Timeout error — `httpx.TimeoutException`

A timeout means the request did not complete within the allowed time.

The normal setting was:

```python
timeout=60.0
```

For the test, the timeout was temporarily changed to:

```python
timeout=0.000001
```

This extremely short value was deliberately unrealistic and used only to trigger the timeout handler.

The terminal showed the timeout message and returned `None`.

**Restore `timeout=60.0` after the test.**

## 15.4 HTTP status error — `httpx.HTTPStatusError`

An HTTP status error means the server returned an unsuccessful HTTP status.

Examples:
- `401`: Unauthorized.
- `404`: Not Found.
- `500`: Internal Server Error.

We used `response.raise_for_status()` to raise an exception for unsuccessful status codes.

For the test, the endpoint was temporarily changed to a deliberately invalid Groq path:

`https://api.groq.com/openai/v1/this-endpoint-does-not-exist`

The terminal displayed the HTTP error, status and response details, then returned `None`.

**Restore the original Groq endpoint after the test.**

## 15.5 The error-handling code

```python
except httpx.ConnectError:
    print("Could not connect to Groq.")

except httpx.TimeoutException:
    print("The request took too long.")

except httpx.HTTPStatusError as error:
    print("Groq returned an HTTP error:")
    print("Status:", error.response.status_code)
    print("Details:", error.response.text)

except (KeyError, IndexError, TypeError, ValueError) as error:
    print("Could not process the model response:", error)

return None
```

The final handler catches some errors that can happen while accessing or converting response data. It is separate from network and HTTP status failures.

# Section 16 — Restore settings and verify the normal workflow

After the three error tests, we restored the original working configuration:

- Endpoint: `https://api.groq.com/openai/v1/chat/completions`
- Timeout: `60.0`
- Model: `openai/gpt-oss-20b`

A final successful test was run using `python harbour.py`. It returned the model output, token counts and latency, confirming that the normal successful workflow still worked after the temporary error tests.

Example final output values:
- Input tokens: 81
- Output tokens: 506
- Total tokens: 587
- Latency: approximately 1.57 seconds

The exact values will vary on another run.

# Section 17 — What does “provider SDK” mean?

This was the final question raised in the lesson.

## 17.1 SDK definition

SDK stands for **Software Development Kit**.

An SDK is a collection of tools, libraries and code interfaces provided to help developers build software that works with a particular platform or service.

A provider SDK is the SDK supplied for working with a provider's service.

Examples of AI providers include Groq, OpenAI and Google. Providers may offer SDKs for their APIs.

## 17.2 Direct HTTP with `httpx` vs a provider SDK

We used direct HTTP with `httpx`. That means we manually:
1. prepared the request body;
2. supplied the authentication header;
3. selected the endpoint;
4. sent the HTTP request;
5. decoded the response;
6. extracted the output and token counts.

A provider SDK usually gives you a client object and methods that handle much of the API communication.

### Direct HTTP example used in the lesson

```python
response = httpx.post(
    "https://api.groq.com/openai/v1/chat/completions",
    headers=headers,
    json=request_data,
    timeout=60.0
)

data = response.json()
output = data["choices"][0]["message"]["content"]
```

### Groq SDK example preview

The following is a conceptual preview of the SDK approach. It was **not yet installed or run during the lessons captured in this notebook**.

```python
from groq import Groq

client = Groq(api_key=API_KEY)

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "My account is locked."
        }
    ]
)

output = response.choices[0].message.content
```

The SDK gives us a `Groq` client and a method for creating a chat completion. We no longer manually write the HTTP endpoint and headers in this call.

## 17.3 Have we completed the exact SDK requirement?

| Requirement | Status |
|---|---|
| Make a model call | Completed with direct HTTP |
| Record input | Completed |
| Record output | Completed |
| Measure latency | Completed |
| Record token usage | Completed |
| Save a JSONL record | Completed |
| Handle and test common errors | Completed |
| Use a provider SDK | Explained, but not yet practised |

**Conclusion:** The model-call, timing and logging workflow has been built and tested. To satisfy the wording “with a provider SDK” literally, the next lesson should install and use the Groq Python SDK, then reproduce the same logging workflow. The team's eventual project provider decision remains Ollama.

# Section 18 — Revision questions

Try to answer these without looking back. You can write your answers in the empty cell below.

1. What does API stand for, and what is its purpose?
2. What is an endpoint?
3. What is the difference between a request and a response?
4. Why did we use HTTP `POST` for our model request?
5. What does JSON stand for?
6. What is the difference between a Python dictionary and JSON text?
7. What does `response.json()` do?
8. What is the difference between `print()` and `return`?
9. What does `time.perf_counter()` help us measure?
10. What is latency?
11. What is an input token? What is an output token?
12. What does `json.dumps()` do?
13. What does JSONL mean?
14. What is the difference between opening a file with `"a"` and `"w"`?
15. What does `response.raise_for_status()` do?
16. What is the difference between `ConnectError` and `TimeoutException`?
17. Why do we keep API keys in `.env` instead of writing them directly in the Python script?
18. What is an SDK?
19. How is direct HTTP with `httpx` different from using a provider SDK?
20. Which provider did the team choose, and which provider did we use temporarily for practice?

## Answer-writing cell

Use this cell to write your own answers. It is intentionally left blank.

In [ ]:
# My revision answers

1.
2.
3.
4.
5.
6.
7.
8.
9.
10.
11.
12.
13.
14.
15.
16.
17.
18.
19.
20.

# Final progress summary — Week 5 Member 8

## Completed and practised

- AI models, Ollama and local vs hosted models.
- APIs, requests, responses and endpoints.
- JSON, Python dictionaries and JSON conversion.
- HTTP, POST and `httpx`.
- Ollama chat request structure and provider-specific response fields.
- Tokens and token usage.
- Latency and `time.perf_counter()`.
- Extracting nested response values.
- Reusable Python functions.
- Environment variables, `.env` and `.gitignore`.
- Groq model call using `httpx`.
- JSONL log creation, append behaviour and reading records.
- Multiple-call testing.
- Connection, timeout and HTTP status error handling.
- Final successful-call verification.
- Meaning of provider SDK and comparison with direct HTTP.

## Still to do

1. Practise the same model call using the actual Groq Python SDK.
2. Reproduce input/output, token, latency and JSONL logging with that SDK.
3. Keep the Groq practice separate from the team's eventual Ollama integration.

## Friday task checklist

- [x] Wrap a model call with timing.
- [x] Record input and output.
- [x] Capture token usage.
- [x] Save a JSON log record.
- [x] Test multiple calls.
- [x] Test common request failures.
- [ ] Repeat the workflow using a provider SDK to match the assignment wording exactly.

**Teaching reminder:** Explain the concepts first, show the request and response, walk through the function in small pieces, run a successful call, inspect the JSONL log, and then demonstrate error handling. This follows the sequence used while learning the topic.